# Module 9 · 文字 4：LLM 訓練資料格式與資料清理

> **本節定位｜2026 主流核心**
> 要訓練/微調**生成式大語言模型 (LLM)**，資料不再是「一句一向量」，
> 而是**有結構的文字資料集**：純文字語料、instruction/chat 的 JSONL、偏好對 (preference)。
> 而且「**資料品質決定模型品質**」——去重、過濾、去污染是現代資料前處理的重頭戲。

## 學習目標
- 認識 LLM 三大訓練階段對應的**資料格式**：預訓練 / 指令微調 (SFT) / 偏好對齊。
- 掌握 **chat template**：`messages` → 套版 → `input_ids`。
- 用 **JSONL** 實作指令資料集（業界標準格式）。
- 動手做**資料清理**：精確去重、品質過濾、長度/語言過濾的直覺與最小實作。
- 理解 **sequence packing** 為什麼能提升預訓練效率。

## 0. 資料結構設計：三種訓練階段、三種資料格式

| 階段 | 目標 | 資料格式 | 一筆樣本長相 |
|:--|:--|:--|:--|
| **預訓練 (Pretrain)** | 學語言/世界知識 | 海量純文字 | `{"text": "..."}`（再 packing 成定長 token 串） |
| **指令微調 (SFT)** | 學會遵循指令/對話 | instruction 或 chat | `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` |
| **偏好對齊 (DPO/RLHF)** | 對齊人類偏好 | 偏好對 | `{"prompt": "...", "chosen": "...", "rejected": "..."}` |

共通點：最終都會被 tokenizer 攤平成 `input_ids (B, L)`（回到 `02` 的標準介面）。
差別在**高層的 schema**，這決定了你怎麼蒐集與清理資料。

In [ ]:
import json, hashlib, re
try:
    from transformers import AutoTokenizer
    HAS_HF = True
except Exception:
    HAS_HF = False
    print("未安裝 transformers，chat template 段落將略過；其餘純 Python 段落可正常執行。")

## 1. Instruction / Chat 格式

現代對話模型的事實標準是 **chat messages**：一個 `role`/`content` 的列表。
`role` 通常是 `system` / `user` / `assistant`。

In [ ]:
chat_sample = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "什麼是 tokenization？"},
        {"role": "assistant", "content": "Tokenization 是把文字切成 subword 並轉成整數 id 的步驟。"},
    ]
}
print(json.dumps(chat_sample, ensure_ascii=False, indent=2))

## 2. Chat Template：把 messages 攤平成模型輸入

每個對話模型都有自己的「對話模板」（用什麼特殊 token 標記角色邊界）。
`tokenizer.apply_chat_template` 會自動套用正確模板，**不要自己手刻**。

In [ ]:
if HAS_HF:
    # 使用一個帶 chat template 的小模型 tokenizer（僅下載 tokenizer 設定，不含權重）
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    prompt_text = tok.apply_chat_template(
        chat_sample["messages"], tokenize=False, add_generation_prompt=False
    )
    print("套用 chat template 後的字串：\n")
    print(prompt_text)
    ids = tok.apply_chat_template(chat_sample["messages"], tokenize=True)
    print(f"\ntokenize 後長度 = {len(ids)} tokens → 最終仍是 input_ids 整數序列")

## 3. JSONL：指令資料集的標準儲存格式

**JSONL**（每行一個 JSON 物件）是 LLM 資料集的事實標準：可串流、可逐行處理、易追加。
大規模時改用 **Parquet**（欄式壓縮、讀取快）。

In [ ]:
dataset = [
    {"instruction": "把這句翻成英文", "input": "今天天氣很好", "output": "The weather is nice today."},
    {"instruction": "用一句話解釋什麼是 BPE", "input": "", "output": "BPE 反覆合併最常見的相鄰字元對來建立 subword 詞彙表。"},
    {"instruction": "把這句翻成英文", "input": "今天天氣很好", "output": "The weather is nice today."},  # 故意重複
]

# 寫成 JSONL（記憶體示意；實務寫檔）
jsonl_text = "\n".join(json.dumps(r, ensure_ascii=False) for r in dataset)
print("JSONL（每行一筆）：")
print(jsonl_text)

## 4. 資料清理：去重、品質過濾

> 「**Garbage in, garbage out**」在 LLM 時代被放大：重複資料會讓模型過擬合、
> 低品質資料會污染能力。以下是最常見的三道清理工序（此處用最小實作示意，
> 大規模會用 MinHash/LSH 做**近似去重**）。

In [ ]:
# (a) 精確去重：用內容雜湊
def row_hash(r):
    key = json.dumps(r, ensure_ascii=False, sort_keys=True)
    return hashlib.md5(key.encode("utf-8")).hexdigest()

seen, deduped = set(), []
for r in dataset:
    h = row_hash(r)
    if h not in seen:
        seen.add(h); deduped.append(r)
print(f"去重前 {len(dataset)} 筆 → 去重後 {len(deduped)} 筆")

# (b) 品質過濾：長度與空輸出等啟發式規則
def is_good(r):
    out = r.get("output", "")
    if len(out.strip()) < 5:            # 太短
        return False
    if r.get("instruction", "").strip() == "":  # 沒有指令
        return False
    return True

filtered = [r for r in deduped if is_good(r)]
print(f"品質過濾後 = {len(filtered)} 筆")
print("\n大規模常用的其他規則：語言偵測、重複 n-gram 比例、URL/亂碼比例、")
print("毒性過濾、以及對測試集的『去污染 (decontamination)』。")

## 5. Sequence Packing：預訓練的效率關鍵

預訓練時若每筆獨立 padding 會浪費大量算力在 padding 上。**packing** 把多筆短文本
用結束符 (EOS) 串接、切成等長 block，幾乎不浪費 token。

In [ ]:
if HAS_HF:
    eos = tok.eos_token_id
    docs = ["短文一。", "另一段稍長一點的文字。", "第三段。"]
    stream = []
    for d in docs:
        stream += tok(d)["input_ids"] + [eos]      # 文件間以 EOS 分隔
    block = 8
    packed = [stream[i:i+block] for i in range(0, len(stream), block)]
    print(f"串接後總 token = {len(stream)}")
    print(f"切成 block_size={block} 的等長區塊數 = {len(packed)}")
    print("每個 block 都是滿的（最後一塊可能不足，實務會丟棄或補齊）→ 幾乎零 padding 浪費。")

## 小結

| 階段 | 格式 | 清理重點 |
|:--|:--|:--|
| 預訓練 | `{"text": ...}` + packing | 去重、品質/語言過濾、去污染 |
| SFT | `messages` / instruction JSONL | chat template、格式一致性 |
| 偏好對齊 | `{prompt, chosen, rejected}` | 配對品質 |

**資料品質 > 資料數量**，是 2026 LLM 資料工程的共識。

**下一節 `05_imdb_case`**：把 `01`（經典）與 `02–03`（現代）整合到一個真實案例，
比較 TF-IDF 與句向量在情感分類上的差異。